#### 1、什么是Tool
Tool（工具）是智能体可以调用的外部函数，让AI能够执行特定任务，如查询天气、搜索信息、计算数据等。

**关键特征**：

- 有明确的输入参数
- 执行具体功能
- 返回结构化结果
- 可以被多个智能体复用



 #### 2、为什么要用Tool

1. **扩展能力**：弥补AI的局限性
2. **获取实时数据**：访问最新信息
3. **执行操作**：调用外部系统API
4. **确保准确性**：减少AI幻觉
5. **标准化流程**：统一业务逻辑

#### 3、如何使用Tool

使用**Responses API** 携带工具

In [9]:
from openai import OpenAI
from  dotenv import load_dotenv
import os

# 加载环境变量
load_dotenv()

# 创建客户端 - 需要传入api_key和base_url
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

# API调用（创建响应）
# response = client.responses.create(
#     model=os.getenv("OPENAI_MODEL_NAME"),
#     tools=[{"type": "web_search"}],
#     input="今天有什么积极的新闻吗?"
# )
# print(response.output_text)


client.chat.completions.create(
    # model=os.getenv("SF_MODEL_NAME"),
    model=os.getenv("OPENAI_MODEL_NAME"),
    messages=[
        {"role": "system", "content": "你是一个搜索专家"},
        {
            "role": "user",
            "content": "今天有什么积极的新闻吗？",
        },
    ],
    tools=[{"type": "web_search"}],
)

# output---message类型的output_text类型的输出项

BadRequestError: Error code: 400 - {'error': {'message': "Invalid value: 'web_search'. Supported values are: 'function' and 'custom'.", 'type': 'invalid_request_error', 'param': 'tools[0].type', 'code': 'invalid_value'}}

这里注意，传统 **Chat Completions API使用不能这样使用**。它的type必须是funcation `“type": "funcation"`不能是工具的名字。

官网链接：https://platform.openai.com/docs/guides/tools?tool-type=function-calling

In [12]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

# 加载环境变量
load_dotenv()

# 创建客户端
client = OpenAI(
    api_key=os.getenv("AL_BAILIAN_API_KEY"),
    base_url=os.getenv("AL_BAILIAN_BASE_URL"),
)


# 定义工具函数
def get_weather(city: str) -> str:
    """模拟查询天气函数"""
    weather_data = {
        "北京": "北京：晴天，15-25°C，适宜外出",
        "上海": "上海：多云，18-28°C，微风",
        "广州": "广州：阵雨，22-30°C，记得带伞",
        "深圳": "深圳：晴转多云，23-31°C，湿度较高"
    }

    return weather_data.get(city, f"暂无{city}的天气信息")


# 使用底层JSON方式定义工具
def chat_with_basic_tools():
    print("=== 1. 底层方式：手动编写工具定义 ===")

    messages = [
        {"role": "system", "content": "你是一个天气助手，可以查询天气信息。"},
        {"role": "user", "content": "我想知道北京的天气怎么样？"}
    ]

    response = client.chat.completions.create(
        model=os.getenv("AL_BAILIAN_MODEL_NAME"),
        messages=messages,
        tools=[{
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "查询指定城市的天气信息",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {
                            "type": "string",
                            "description": "城市名称，如'北京'、'上海'",
                        },
                    },
                    "required": ["city"],
                    "additionalProperties": False
                },
            }
        }],
    )
    print(response)

    message = response.choices[0].message
    messages.append(message)
    #
    if message.tool_calls:
        for tool_call in message.tool_calls:
            if tool_call.function.name == "get_weather":
                args = json.loads(tool_call.function.arguments)
                result = get_weather(args["city"]) # 调用工具
                print(result)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })

        second_response = client.chat.completions.create(
            model=os.getenv("AL_BAILIAN_MODEL_NAME"),
            messages=messages,
        )
        print(f"最终回复: {second_response.choices[0].message.content}")

    return response


if __name__ == "__main__":
    chat_with_basic_tools()


=== 1. 底层方式：手动编写工具定义 ===
ChatCompletion(id='chatcmpl-95dd3237-ff95-95d0-b419-e4a8bb08ce24', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_24b78aba244e4aa5aba02817', function=Function(arguments='{"city": "北京"}', name='get_weather'), type='function', index=0)]))], created=1768549149, model='qwen3-max', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=21, prompt_tokens=290, total_tokens=311, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0)))
xxxxxxxxxxx
北京：晴天，15-25°C，适宜外出
最终回复: 北京今天是晴天，气温在15到25摄氏度之间，天气舒适，非常适合外出活动！


In [16]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv(override=True)


client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

def get_weather(city: str) -> str:
    """模拟查询天气函数"""
    weather_data = {
        "北京": "北京：晴天，15-25°C，适宜外出",
        "上海": "上海：多云，18-28°C，微风",
        "广州": "广州：阵雨，22-30°C，记得带伞",
        "深圳": "深圳：晴转多云，23-31°C，湿度较高"
    }
    print("xxxxxxxxxxx")
    return weather_data.get(city, f"暂无{city}的天气信息")



tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country e.g. Bogotá, Colombia",
                }
            },
            "required": ["location"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

response = client.responses.create(
    model="gpt-5",
    input=[
        {"role": "user", "content": "What is the weather like in Paris today?"},
    ],
    tools=tools,
)

print(response)

NotFoundError: 404 page not found

#### 4、Tool使用场景

| 场景     | 工具示例                              | 说明           |
| :------- | :------------------------------------ | :------------- |
| 数据查询 | `search_product`, `query_order`       | 查询数据库信息 |
| 计算服务 | `calculate_price`, `convert_currency` | 执行数学计算   |
| 外部API  | `send_email`, `create_ticket`         | 调用第三方服务 |
| 文件操作 | `read_file`, `generate_report`        | 处理文档数据   |

- Response API 在工具处理上属于半自动化(在线工具上)---找工具--->调用工具（自定义的工具）== Chat completions（等价的）
- Chat completions 在工具处理上属于手动（自定调用工具）模型只找到工具。

In [14]:
import os
from dotenv import load_dotenv


load_dotenv(override=True)

print("OPENAI_BASE_URL =", os.getenv("OPENAI_BASE_URL"))
print("OPENAI_API_KEY exists =", os.getenv("OPENAI_API_KEY") is not None)
print("OPENAI_API_KEY prefix =", os.getenv("OPENAI_API_KEY")[:10] if os.getenv("OPENAI_API_KEY") else None)

OPENAI_BASE_URL = https://api.openai.com/v1
OPENAI_API_KEY exists = True
OPENAI_API_KEY prefix = sk-proj-SI
